In [8]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image


# --- 3. 初始化检测器 (之前代码漏掉的部分) ---
model_path = 'pose_landmarker.task' 
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO
)
detector = vision.PoseLandmarker.create_from_options(options)



In [10]:
import math

def calculate_angle(a, b, c):
    """
    计算三个点之间的夹角 (单位：度)
    a, b, c 分别是 landmarks 列表中的点对象
    """
    # 转换为 numpy 数组方便计算 (这里用 x, y 即可，如果需要 3D 角度则加入 z)
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])
    
    # 计算向量
    ba = a - b
    bc = c - b
    
    # 使用余弦定理公式计算弧度
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    # 防止浮点数精度问题导致超出 [-1, 1] 范围
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    
    angle = np.degrees(np.arccos(cosine_angle))
    return angle

In [7]:
#显示24,25,26的角度
cap = cv2.VideoCapture(2)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 1. 基础处理
    frame = cv2.flip(frame, 1) # 镜像翻转
    h, w, _ = frame.shape
    
       # 1. 准备数据
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    # 获取当前时间戳（MediaPipe 视频模式必需）
    timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    
    # 2. 识别姿态
    global last_result
    last_result = detector.detect_for_video(mp_image, timestamp)
    
    # 3. 提取右腿并计算角度
    if last_result.pose_landmarks:
        landmarks = last_result.pose_landmarks[0]
        
        # 右腿索引：24(髋), 26(膝), 28(踝)
        hip = landmarks[24]
        knee = landmarks[26]
        ankle = landmarks[28]

        # 计算膝盖弯曲角度
        angle = calculate_angle(hip, knee, ankle)

        # 4. 可视化：绘制骨架和角度文字
        # 先用官方函数画全画面的骨架线
        frame_with_skeleton = draw_landmarks_on_image(frame_rgb, last_result)
        frame = cv2.cvtColor(frame_with_skeleton, cv2.COLOR_RGB2BGR)

        # 在膝盖位置标出度数
        knee_x, knee_y = int(knee.x * w), int(knee.y * h)
        
        # 绘制一个半透明背景条，让数字更清晰
        cv2.rectangle(frame, (knee_x + 15, knee_y - 25), (knee_x + 130, knee_y + 10), (0, 0, 0), -1)
        cv2.putText(frame, f"{int(angle)} DEG", (knee_x + 20, knee_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)

    # 5. 显示窗口
    cv2.imshow('Right Leg Kick Angle Analysis', frame)
    
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

In [12]:
#计数
# 初始化全局变量
counter = 0 
stage = "READY"  # 初始状态
max_angle_in_swing = 0 # 记录单次摆腿的最大角度

cap = cv2.VideoCapture(2)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    
    last_result = detector.detect_for_video(mp_image, timestamp)
    
    if last_result.pose_landmarks:
        landmarks = last_result.pose_landmarks[0]
        
        # 1. 计算右腿膝盖角度 (24:髋, 26:膝, 28:踝)
        angle = calculate_angle(landmarks[24], landmarks[26], landmarks[28])

        # 2. 【踢球计数核心逻辑】
        # 科学阈值：后摆弯曲 < 100度，击球伸直 > 160度
        
        # 阶段 A：检测到后摆蓄力（膝盖弯曲）
        if angle < 100:
            stage = "BACKSWING" # 后摆阶段
        
        # 阶段 B：检测到向前踢出（腿部伸直）
        # 必须先经历过 BACKSWING 阶段，才能触发计数
        if angle > 160 and stage == "BACKSWING":
            stage = "KICK" # 击球完成
            counter += 1
            print(f"有效踢球！当前计数: {counter}, 击球角度: {int(angle)}")

        # 3. 绘图与显示
        annotated_frame_rgb = draw_landmarks_on_image(frame_rgb, last_result)
        frame = cv2.cvtColor(annotated_frame_rgb, cv2.COLOR_RGB2BGR)

        # UI 显示：左上角状态栏
        cv2.rectangle(frame, (0,0), (280, 80), (0,0,0), -1)
        cv2.putText(frame, f"STAGE: {stage}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 1, cv2.LINE_AA)
        cv2.putText(frame, f"KICK COUNT: {counter}", (10, 70), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
        
        # 在右膝盖位置实时显示角度
        knee_pos = (int(landmarks[26].x * w), int(landmarks[26].y * h))
        cv2.putText(frame, f"{int(angle)} DEG", (knee_pos[0]+20, knee_pos[1]), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)

    cv2.imshow('Football Kick Analyzer', frame)
    
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

有效踢球！当前计数: 1, 击球角度: 163
有效踢球！当前计数: 2, 击球角度: 173
有效踢球！当前计数: 3, 击球角度: 161
有效踢球！当前计数: 4, 击球角度: 167
有效踢球！当前计数: 5, 击球角度: 160
有效踢球！当前计数: 6, 击球角度: 162
有效踢球！当前计数: 7, 击球角度: 176


In [14]:
#蓄力记录

# --- 1. 初始化计数与蓄力变量 ---
counter = 0 
stage = "READY" 
min_angle_this_kick = 180.0  # 记录单次摆腿中达到的最小角度
last_kick_quality = 0        # 记录上一脚的蓄力数据用于显示

# --- 2. 初始化激活与稳定性变量 ---
is_active = False          # 是否激活识别
alignment_counter = 0      # 对齐计数器
REQUIRED_FRAMES = 30       # 需要连续稳定识别 30 帧（约 1 秒）

cap = cv2.VideoCapture(2)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    # 准备 MediaPipe 数据
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    timestamp = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    
    # 姿态识别
    last_result = detector.detect_for_video(mp_image, timestamp)
    
    if last_result.pose_landmarks:
        landmarks = last_result.pose_landmarks[0]
        
        # 提取关键点可见度
        hip_v = landmarks[24].visibility
        knee_v = landmarks[26].visibility
        ankle_v = landmarks[28].visibility

        # --- 3. 站定激活逻辑 ---
        if not is_active:
            # 只有当髋、膝、踝都在画面内且清晰时才累加
            if hip_v > 0.7 and knee_v > 0.7 and ankle_v > 0.7:
                alignment_counter += 1
                status_text = f"Aligning... {int((alignment_counter/REQUIRED_FRAMES)*100)}%"
            else:
                alignment_counter = 0
                status_text = "Waiting for Full Leg View..."

            if alignment_counter >= REQUIRED_FRAMES:
                is_active = True
                print("系统已激活，可以开始踢球！")
        
        # --- 4. 激活后的计数与分析逻辑 ---
        else:
            status_text = "SYSTEM ACTIVE"
            # 计算右腿角度
            angle = calculate_angle(landmarks[24], landmarks[26], landmarks[28])

            # A. 蓄力检测 (Backswing)
            if angle < 110:
                if stage != "BACKSWING":
                    stage = "BACKSWING"
                    min_angle_this_kick = 180.0
                if angle < min_angle_this_kick:
                    min_angle_this_kick = angle

            # B. 击球检测 (Impact)
            elif angle > 160 and stage == "BACKSWING":
                stage = "KICK_COMPLETE"
                counter += 1
                last_kick_quality = int(min_angle_this_kick)
            
            # C. 状态复位
            elif 130 < angle < 150 and stage == "KICK_COMPLETE":
                stage = "READY"

            # 丢失检测：如果踢球用力过猛导致脚飞出镜头，自动重置系统
            if hip_v < 0.5 or ankle_v < 0.5:
                is_active = False
                alignment_counter = 0
                stage = "READY"
                print("目标丢失，请重新站定")

        # --- 5. UI 绘制与显示 ---
        # 绘制背景和骨架
        annotated_frame_rgb = draw_landmarks_on_image(frame_rgb, last_result)
        frame = cv2.cvtColor(annotated_frame_rgb, cv2.COLOR_RGB2BGR)
        
        # 绘制黑底面板
        cv2.rectangle(frame, (0, 0), (320, 160), (0, 0, 0), -1)
        
        # 显示计数和质量数据
        cv2.putText(frame, f"KICKS: {counter}", (10, 40), 1, 2, (0, 255, 0), 2)
        cv2.putText(frame, f"LAST BEND: {last_kick_quality} DEG", (10, 80), 1, 1.2, (0, 255, 255), 2)
        
        # 显示激活状态文字（变色逻辑）
        text_color = (0, 255, 0) if is_active else (0, 0, 255)
        cv2.putText(frame, status_text, (10, 130), 1, 1.2, text_color, 2)

    cv2.imshow('Dongye Football Analyzer v1.0', frame)
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

系统已激活，可以开始踢球！
目标丢失，请重新站定
系统已激活，可以开始踢球！
目标丢失，请重新站定
